In [86]:
import os
import io
import json
import math
import time
import random
from pathlib import Path
from dataclasses import dataclass, asdict

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from diffusers.schedulers import DDIMScheduler

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

In [87]:
@dataclass
class CFG:
    # paths
    root: str = "precomputed_waymo_e2e"
    anchors_path: str = "precomputed_waymo_e2e/anchors/anchors_k20.npy"
    work_dir: str = "precomputed_waymo_e2e/training_runs_20/diffusion_planner_v2_strong"

    # data
    img_h: int = 256
    img_w: int = 512
    use_rgb: bool = True
    use_road: bool = True
    use_lane: bool = True
    use_lane_prob: bool = True
    use_veh: bool = True
    use_depth: bool = True

    future_len: int = 20
    past_len_max: int = 16
    num_anchors: int = 20

    # intent
    use_intent: bool = True
    num_intents: int = 3  # количество верхнеуровневых команд
    intent_emb_dim: int = 32

    # model
    d_model: int = 256
    n_heads: int = 8
    n_decoder_layers: int = 3
    mlp_dim: int = 768
    scene_dim: int = 256
    past_encoder_dim: int = 128
    scene_encoder_layers: int = 2
    scene_dropout: float = 0.1
    past_num_layers: int = 2

    # diffusion
    num_train_timesteps: int = 1000
    train_max_timestep: int = 50   # truncated diffusion
    infer_steps: int = 1
    infer_start_timestep: int = 8

    # loss
    cls_loss_weight: float = 1.0
    reg_loss_weight: float = 2.0
    endpoint_loss_weight: float = 2.0
    soft_target_temperature: float = 2.0
    smooth_l1_beta: float = 1.0

    # train
    seed: int = 42
    batch_size: int = 32
    num_workers: int = 4
    epochs: int = 20
    lr: float = 1e-4
    weight_decay: float = 1e-4
    grad_clip_norm: float = 1.0
    log_every: int = 100
    val_every: int = 1
    mixed_precision: bool = True

cfg = CFG()
Path(cfg.work_dir).mkdir(parents=True, exist_ok=True)

print(asdict(cfg))


{'root': 'precomputed_waymo_e2e', 'anchors_path': 'precomputed_waymo_e2e/anchors/anchors_k20.npy', 'work_dir': 'precomputed_waymo_e2e/training_runs_20/diffusion_planner_v1', 'img_h': 256, 'img_w': 512, 'use_rgb': True, 'use_road': True, 'use_lane': True, 'use_lane_prob': True, 'use_veh': True, 'use_depth': True, 'future_len': 20, 'past_len_max': 16, 'num_anchors': 20, 'use_intent': True, 'num_intents': 3, 'intent_emb_dim': 16, 'd_model': 256, 'n_heads': 8, 'n_decoder_layers': 2, 'mlp_dim': 512, 'scene_dim': 256, 'past_encoder_dim': 128, 'num_train_timesteps': 1000, 'train_max_timestep': 50, 'infer_steps': 2, 'infer_start_timestep': 8, 'cls_loss_weight': 1.0, 'reg_loss_weight': 2.0, 'seed': 42, 'batch_size': 32, 'num_workers': 4, 'epochs': 20, 'lr': 0.0001, 'weight_decay': 0.0001, 'grad_clip_norm': 1.0, 'log_every': 100, 'val_every': 1, 'mixed_precision': True}


In [88]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


In [89]:
import cv2
print(cv2.__version__)

4.13.0


In [90]:
import cv2

def load_rgb(path: Path, size):
    img = Image.open(path).convert("RGB").resize(size, Image.BILINEAR)
    arr = np.asarray(img, dtype=np.float32) / 255.0   # HWC
    arr = np.transpose(arr, (2, 0, 1))                # CHW
    return arr

def load_mask(path: Path, size):
    img = Image.open(path).convert("L").resize(size, Image.NEAREST)
    arr = np.asarray(img, dtype=np.float32) / 255.0
    return arr[None]   # 1,H,W

def load_lane_prob(path: Path, size):
    """
    Загружает lane probability map как raw float map.
    ВАЖНО: без per-sample min-max normalization.
    """
    arr = np.load(path).astype(np.float32)  # H,W
    arr = cv2.resize(arr, size, interpolation=cv2.INTER_LINEAR)  # size=(W,H)
    return arr[None]  # 1,H,W

def load_depth_map(path: Path, size, clip_max=100.0):
    """
    Загружает depth map.
    Предполагаем, что depth в метрах/похожей шкале.
    Делаем clip + normalize в [0,1].
    """
    arr = np.load(path).astype(np.float32)  # H,W
    arr = np.clip(arr, 0.0, clip_max) / clip_max
    arr = cv2.resize(arr, size, interpolation=cv2.INTER_LINEAR)
    return arr[None]  # 1,H,W

def pad_or_trim_past_xy(past_xy, target_len):
    arr = np.asarray(past_xy, dtype=np.float32)
    if len(arr) >= target_len:
        return arr[-target_len:]
    pad = np.zeros((target_len - len(arr), 2), dtype=np.float32)
    return np.concatenate([pad, arr], axis=0)

def future_xy_to_array(future_xy, target_len):
    arr = np.asarray(future_xy, dtype=np.float32)
    assert arr.shape[1] == 2
    if len(arr) != target_len:
        raise ValueError(f"future len mismatch: got {arr.shape}, expected ({target_len},2)")
    return arr

In [91]:
anchors = np.load(cfg.anchors_path).astype(np.float32)
print("anchors shape:", anchors.shape)

assert anchors.shape[0] == cfg.num_anchors
assert anchors.shape[1] == cfg.future_len
assert anchors.shape[2] == 2

anchors shape: (20, 20, 2)


In [92]:
def build_samples_with_anchor_jsonl(split: str, cfg: CFG, force_rebuild: bool = False):
    root = Path(cfg.root)
    split_root = root / split
    index_path = split_root / "index.jsonl"
    out_path = split_root / f"{split}_samples_with_anchor_k{cfg.num_anchors}.jsonl"

    anchors_root = root / "anchors"

    labels_path = anchors_root / f"{split}_anchor_labels_k{cfg.num_anchors}.npy"
    dists_path = anchors_root / f"{split}_anchor_assigned_dist_k{cfg.num_anchors}.npy"
    sample_ids_path = anchors_root / f"{split}_anchor_sample_ids_k{cfg.num_anchors}.json"

    if out_path.exists() and not force_rebuild:
        print("already exists:", out_path)
        return out_path

    if not labels_path.exists():
        print(f"[{split}] no labels file:", labels_path)
        return None
    if not dists_path.exists():
        print(f"[{split}] no dists file:", dists_path)
        return None
    if not sample_ids_path.exists():
        print(f"[{split}] no sample_ids file:", sample_ids_path)
        return None

    labels = np.load(labels_path)
    dists = np.load(dists_path)

    with open(sample_ids_path, "r", encoding="utf-8") as f:
        anchor_sample_ids = json.load(f)

    if not (len(labels) == len(dists) == len(anchor_sample_ids)):
        raise ValueError(
            f"[{split}] mismatch: len(labels)={len(labels)}, "
            f"len(dists)={len(dists)}, len(sample_ids)={len(anchor_sample_ids)}"
        )

    anchor_map = {
        str(sample_id): (int(labels[i]), float(dists[i]))
        for i, sample_id in enumerate(anchor_sample_ids)
    }

    rows = []
    skipped_missing_meta = 0
    skipped_no_anchor = 0

    with open(index_path, "r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            sample_dir = split_root / row["sample_dir"]
            meta_path = sample_dir / "meta.json"

            if not meta_path.exists():
                skipped_missing_meta += 1
                continue

            with open(meta_path, "r", encoding="utf-8") as mf:
                meta = json.load(mf)

            sample_id = str(meta["sample_id"])

            if sample_id not in anchor_map:
                skipped_no_anchor += 1
                continue

            anchor_id, anchor_dist = anchor_map[sample_id]

            merged = {
                "sample_id": meta["sample_id"],
                "split": split,
                "sample_dir": row["sample_dir"],
                "context_name": row.get("context_name", meta.get("context_name")),
                "intent": meta["intent"],
                "past_len": len(meta["past_xy"]),
                "future_len": len(meta["future_xy"]),
                "anchor_id": anchor_id,
                "anchor_dist": anchor_dist,
            }
            rows.append(merged)

    with open(out_path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    print(f"[{split}] saved:", out_path)
    print(f"[{split}] rows:", len(rows))
    print(f"[{split}] skipped_missing_meta:", skipped_missing_meta)
    print(f"[{split}] skipped_no_anchor:", skipped_no_anchor)

    return out_path

train_jsonl = build_samples_with_anchor_jsonl("train", cfg, force_rebuild=True)
val_jsonl = build_samples_with_anchor_jsonl("val", cfg, force_rebuild=True)

[train] saved: precomputed_waymo_e2e/train/train_samples_with_anchor_k20.jsonl
[train] rows: 415663
[train] skipped_missing_meta: 0
[train] skipped_no_anchor: 0
[val] no labels file: precomputed_waymo_e2e/anchors/val_anchor_labels_k20.npy


In [93]:
from collections import Counter
import json

def inspect_intents(jsonl_path):
    intents = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            intents.append(row["intent"])

    print("total samples:", len(intents))
    print("unique count:", len(set(intents)))
    print("unique sorted:", sorted(set(intents)))
    print("counter:", Counter(intents))

    try:
        vals = sorted(set(int(x) for x in intents))
        print("as int sorted:", vals)
        print("min:", min(vals))
        print("max:", max(vals))
        print("contiguous 0..N-1:", vals == list(range(len(vals))))
        print("recommended cfg.num_intents =", len(vals))
    except Exception as e:
        print("int cast failed:", e)

inspect_intents(train_jsonl)

total samples: 415663
unique count: 3
unique sorted: [1, 2, 3]
counter: Counter({1: 349956, 2: 35837, 3: 29870})
as int sorted: [1, 2, 3]
min: 1
max: 3
contiguous 0..N-1: False
recommended cfg.num_intents = 3


In [94]:
def intent_distribution_from_jsonl(path):
    counter = {}
    total = 0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            x = int(row["intent"])
            counter[x] = counter.get(x, 0) + 1
            total += 1
    return counter, total

for name, path in [("train", train_jsonl), ("val", val_jsonl)]:
    if path is None:
        print(f"\n{name}: skipped (no jsonl)")
        continue

    counter, total = intent_distribution_from_jsonl(path)
    print(f"\n{name}: total={total}")
    for k, v in sorted(counter.items()):
        print(f"intent={k}: {v} ({100*v/total:.2f}%)")


train: total=415663
intent=1: 349956 (84.19%)
intent=2: 35837 (8.62%)
intent=3: 29870 (7.19%)

val: skipped (no jsonl)


In [95]:
def build_samples_jsonl(split: str, cfg: CFG, force_rebuild: bool = False):
    root = Path(cfg.root)
    split_root = root / split
    index_path = split_root / "index.jsonl"
    out_path = split_root / f"{split}_samples.jsonl"

    if out_path.exists() and not force_rebuild:
        print("already exists:", out_path)
        return out_path

    rows = []
    skipped_missing_meta = 0
    skipped_missing_intent = 0

    with open(index_path, "r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            sample_dir = split_root / row["sample_dir"]
            meta_path = sample_dir / "meta.json"

            if not meta_path.exists():
                skipped_missing_meta += 1
                continue

            with open(meta_path, "r", encoding="utf-8") as mf:
                meta = json.load(mf)

            if "intent" not in meta:
                skipped_missing_intent += 1
                continue

            merged = {
                "sample_id": meta["sample_id"],
                "split": split,
                "sample_dir": row["sample_dir"],
                "context_name": row.get("context_name", meta.get("context_name")),
                "intent": int(meta["intent"]),
                "past_len": len(meta["past_xy"]),
                "future_len": len(meta["future_xy"]),
            }
            rows.append(merged)

    with open(out_path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    print(f"[{split}] saved:", out_path)
    print(f"[{split}] rows:", len(rows))
    print(f"[{split}] skipped_missing_meta:", skipped_missing_meta)
    print(f"[{split}] skipped_missing_intent:", skipped_missing_intent)

    return out_path

In [96]:
train_jsonl = build_samples_with_anchor_jsonl("train", cfg, force_rebuild=True)
val_jsonl = build_samples_jsonl("val", cfg, force_rebuild=True)

[train] saved: precomputed_waymo_e2e/train/train_samples_with_anchor_k20.jsonl
[train] rows: 415663
[train] skipped_missing_meta: 0
[train] skipped_no_anchor: 0
[val] saved: precomputed_waymo_e2e/val/val_samples.jsonl
[val] rows: 106360
[val] skipped_missing_meta: 0
[val] skipped_missing_intent: 0


In [144]:
from pathlib import Path
import json
import numpy as np
import torch
from torch.utils.data import Dataset
from tqdm.auto import tqdm


class WaymoPrecomputedPlannerDataset(Dataset):
    def __init__(self, split: str, jsonl_path: Path, cfg: CFG):
        self.split = split
        self.root = Path(cfg.root)
        self.split_root = self.root / split
        self.cfg = cfg
        self.size = (cfg.img_w, cfg.img_h)  # PIL/cv2 use (W,H)
        self.intent_map = {1: 0, 2: 1, 3: 2}

        base_rows = []
        with open(jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                base_rows.append(json.loads(line))

        self.rows = []
        for row in tqdm(base_rows, desc=f"index {split} dataset"):
            sample_dir = self.split_root / row["sample_dir"]
            meta_path = sample_dir / "meta.json"

            with open(meta_path, "r", encoding="utf-8") as f:
                meta = json.load(f)

            raw_intent = int(meta["intent"])
            intent = self.intent_map[raw_intent]

            indexed = {
                "sample_dir": sample_dir,
                "past_xy": pad_or_trim_past_xy(meta["past_xy"], self.cfg.past_len_max),
                "future_xy": future_xy_to_array(meta["future_xy"], self.cfg.future_len),
                "intent": intent,
                "sample_id": int(meta["sample_id"]),
            }

            if self.cfg.use_rgb:
                indexed["rgb_path"] = sample_dir / meta["rgb_path"]
            if self.cfg.use_road:
                indexed["road_path"] = sample_dir / meta["road_path"]
            if self.cfg.use_lane:
                indexed["lane_path"] = sample_dir / meta["lane_path"]
            if self.cfg.use_veh:
                indexed["veh_path"] = sample_dir / meta["veh_path"]
            if self.cfg.use_lane_prob:
                indexed["lane_prob_path"] = sample_dir / meta["lane_prob_path"]
            if self.cfg.use_depth:
                indexed["depth_path"] = sample_dir / meta["depth_path"]

            if "anchor_id" in row:
                indexed["anchor_id"] = int(row["anchor_id"])
            if "anchor_dist" in row:
                indexed["anchor_dist"] = float(row["anchor_dist"])

            self.rows.append(indexed)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]

        channels = []

        if self.cfg.use_rgb:
            channels.append(load_rgb(row["rgb_path"], self.size))
        if self.cfg.use_road:
            channels.append(load_mask(row["road_path"], self.size))
        if self.cfg.use_lane:
            channels.append(load_mask(row["lane_path"], self.size))
        if self.cfg.use_veh:
            channels.append(load_mask(row["veh_path"], self.size))
        if self.cfg.use_lane_prob:
            channels.append(load_lane_prob(row["lane_prob_path"], self.size))
        if self.cfg.use_depth:
            channels.append(load_depth_map(row["depth_path"], self.size))

        x = np.concatenate(channels, axis=0).astype(np.float32)

        item = {
            "x": torch.from_numpy(x),
            "past_xy": torch.from_numpy(row["past_xy"]).float(),
            "future_xy": torch.from_numpy(row["future_xy"]).float(),
            "intent": torch.tensor(row["intent"], dtype=torch.long),
            "sample_id": torch.tensor(row["sample_id"], dtype=torch.long),
        }

        if "anchor_id" in row:
            item["anchor_id"] = torch.tensor(row["anchor_id"], dtype=torch.long)
        if "anchor_dist" in row:
            item["anchor_dist"] = torch.tensor(row["anchor_dist"], dtype=torch.float32)

        return item

In [145]:
train_ds = WaymoPrecomputedPlannerDataset("train", train_jsonl, cfg)
val_ds = WaymoPrecomputedPlannerDataset("val", val_jsonl, cfg)

print("train size:", len(train_ds))
print("val size:", len(val_ds))

sample = train_ds[0]
for k, v in sample.items():
    if torch.is_tensor(v):
        print(k, tuple(v.shape), v.dtype)
    else:
        print(k, type(v))

print("\nInput channel stats:")
x = sample["x"].numpy()
for i in range(x.shape[0]):
    print(
        f"channel {i:02d}: "
        f"min={x[i].min():.6f} "
        f"max={x[i].max():.6f} "
        f"mean={x[i].mean():.6f} "
        f"std={x[i].std():.6f}"
    )

Traceback (most recent call last):
  File "/usr/lib/python3.10/multiprocessing/queues.py", line 239, in _feed
    reader_close()
  File "/usr/lib/python3.10/multiprocessing/connection.py", line 177, in close
    self._close()
  File "/usr/lib/python3.10/multiprocessing/connection.py", line 361, in _close
    _close(self._handle)
OSError: [Errno 9] Bad file descriptor
index val dataset: 100%|██████████| 106360/106360 [00:14<00:00, 7294.95it/s]


train size: 415663
val size: 106360
x (8, 256, 512) torch.float32
past_xy (16, 2) torch.float32
future_xy (20, 2) torch.float32
intent () torch.int64
sample_id () torch.int64
anchor_id () torch.int64
anchor_dist () torch.float32

Input channel stats:
channel 00: min=0.039216 max=1.000000 mean=0.330746 std=0.195693
channel 01: min=0.023529 max=1.000000 mean=0.343623 std=0.208685
channel 02: min=0.019608 max=1.000000 mean=0.399268 std=0.255703
channel 03: min=0.000000 max=1.000000 mean=0.266785 std=0.442279
channel 04: min=0.000000 max=1.000000 mean=0.677773 std=0.467330
channel 05: min=0.000000 max=1.000000 mean=0.005836 std=0.076174
channel 06: min=0.500000 max=0.730957 mean=0.503484 std=0.027136
channel 07: min=0.052852 max=1.000000 mean=0.534146 std=0.398578


In [146]:
print("train intent sample:", train_ds[0]["intent"].item())
print("val intent sample:", val_ds[0]["intent"].item())

train intent sample: 0
val intent sample: 2


In [147]:
val_sample = val_ds[0]
print("val keys:", list(val_sample.keys()))
print("has anchor_id:", "anchor_id" in val_sample)
print("has anchor_dist:", "anchor_dist" in val_sample)

val keys: ['x', 'past_xy', 'future_xy', 'intent', 'sample_id']
has anchor_id: False
has anchor_dist: False


In [148]:
train_ds = WaymoPrecomputedPlannerDataset("train", train_jsonl, cfg)
val_ds = WaymoPrecomputedPlannerDataset("val", val_jsonl, cfg)

index val dataset: 100%|██████████| 106360/106360 [00:12<00:00, 8513.62it/s] 


In [149]:
print("train keys:", list(train_ds[0].keys()))
print("val keys:", list(val_ds[0].keys()))

train keys: ['x', 'past_xy', 'future_xy', 'intent', 'sample_id', 'anchor_id', 'anchor_dist']
val keys: ['x', 'past_xy', 'future_xy', 'intent', 'sample_id']


In [150]:
def show_dataset_sample(ds, idx=0):
    s = ds[idx]
    x = s["x"].numpy()

    names = []
    channel_ptr = 0

    fig, axes = plt.subplots(2, 4, figsize=(18, 9))
    axes = axes.flatten()

    plot_idx = 0

    if cfg.use_rgb:
        rgb = np.transpose(x[channel_ptr:channel_ptr+3], (1,2,0))
        axes[plot_idx].imshow(rgb)
        axes[plot_idx].set_title("RGB")
        axes[plot_idx].axis("off")
        channel_ptr += 3
        plot_idx += 1

    if cfg.use_road:
        axes[plot_idx].imshow(x[channel_ptr], cmap="gray")
        axes[plot_idx].set_title("road")
        axes[plot_idx].axis("off")
        channel_ptr += 1
        plot_idx += 1

    if cfg.use_lane:
        axes[plot_idx].imshow(x[channel_ptr], cmap="gray")
        axes[plot_idx].set_title("lane")
        axes[plot_idx].axis("off")
        channel_ptr += 1
        plot_idx += 1

    if cfg.use_veh:
        axes[plot_idx].imshow(x[channel_ptr], cmap="gray")
        axes[plot_idx].set_title("veh")
        axes[plot_idx].axis("off")
        channel_ptr += 1
        plot_idx += 1

    if cfg.use_lane_prob:
        axes[plot_idx].imshow(x[channel_ptr], cmap="viridis", vmin=0.5, vmax=0.75)
        axes[plot_idx].set_title("lane_prob")
        axes[plot_idx].axis("off")
        channel_ptr += 1
        plot_idx += 1

    if cfg.use_depth:
        axes[plot_idx].imshow(x[channel_ptr], cmap="magma")
        axes[plot_idx].set_title("depth")
        axes[plot_idx].axis("off")
        channel_ptr += 1
        plot_idx += 1

    # скрыть пустые subplot
    while plot_idx < len(axes):
        axes[plot_idx].axis("off")
        plot_idx += 1

    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(6,6))
    past_xy = s["past_xy"].numpy()
    future_xy = s["future_xy"].numpy()
    anchor = anchors[s["anchor_id"].item()]
    plt.plot(past_xy[:,0], past_xy[:,1], marker="o", label="past")
    plt.plot(future_xy[:,0], future_xy[:,1], marker="o", label="future_gt")
    plt.plot(anchor[:,0], anchor[:,1], marker="o", label=f"anchor_{s['anchor_id'].item()}")
    plt.axhline(0, color="gray", linewidth=1)
    plt.axvline(0, color="gray", linewidth=1)
    plt.grid(True, alpha=0.3)
    plt.axis("equal")
    plt.legend()
    plt.title(
        f"intent={s['intent'].item()} "
        f"sample_id={s['sample_id'].item()} "
        f"anchor_id={s['anchor_id'].item()}"
    )
    plt.show()

    print("Per-channel stats:")
    for i in range(x.shape[0]):
        print(
            f"channel {i:02d}: "
            f"min={x[i].min():.6f} "
            f"max={x[i].max():.6f} "
            f"mean={x[i].mean():.6f} "
            f"std={x[i].std():.6f}"
        )

In [164]:
train_loader = DataLoader(
    train_ds,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=True,
    drop_last=True,
    persistent_workers=(cfg.num_workers > 0),
    prefetch_factor=2 if cfg.num_workers > 0 else None,
)

val_loader = DataLoader(
    val_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=True,
    drop_last=False,
    persistent_workers=(cfg.num_workers > 0),
    prefetch_factor=2 if cfg.num_workers > 0 else None,
)

batch = next(iter(train_loader))
for k, v in batch.items():
    if torch.is_tensor(v):
        print(k, tuple(v.shape), v.dtype)


x (32, 8, 256, 512) torch.float32
past_xy (32, 16, 2) torch.float32
future_xy (32, 20, 2) torch.float32
intent (32,) torch.int64
sample_id (32,) torch.int64
anchor_id (32,) torch.int64
anchor_dist (32,) torch.float32


In [165]:
class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)
        self.drop = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()

        if stride != 1 or in_ch != out_ch:
            self.skip = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )
        else:
            self.skip = nn.Identity()

    def forward(self, x):
        identity = self.skip(x)
        x = self.act(self.bn1(self.conv1(x)))
        x = self.drop(x)
        x = self.bn2(self.conv2(x))
        x = self.act(x + identity)
        return x


class SimpleSceneEncoder(nn.Module):
    def __init__(self, in_ch, cfg: CFG):
        super().__init__()
        self.cfg = cfg

        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )

        self.layer1 = nn.Sequential(
            ResidualBlock(64, 64, stride=1, dropout=cfg.scene_dropout),
            ResidualBlock(64, 64, stride=1, dropout=cfg.scene_dropout),
        )
        self.layer2 = nn.Sequential(
            ResidualBlock(64, 128, stride=2, dropout=cfg.scene_dropout),
            ResidualBlock(128, 128, stride=1, dropout=cfg.scene_dropout),
        )
        self.layer3 = nn.Sequential(
            ResidualBlock(128, 256, stride=2, dropout=cfg.scene_dropout),
            ResidualBlock(256, 256, stride=1, dropout=cfg.scene_dropout),
        )
        self.layer4 = nn.Sequential(
            ResidualBlock(256, 256, stride=2, dropout=cfg.scene_dropout),
            ResidualBlock(256, 256, stride=1, dropout=cfg.scene_dropout),
        )

        self.pool = nn.AdaptiveAvgPool2d((8, 8))
        self.proj = nn.Conv2d(256, cfg.d_model, kernel_size=1)

        self.bev_pos_emb = nn.Parameter(torch.zeros(1, 64, cfg.d_model))
        enc_layer = nn.TransformerEncoderLayer(
            d_model=cfg.d_model,
            nhead=cfg.n_heads,
            dim_feedforward=cfg.mlp_dim,
            dropout=cfg.scene_dropout,
            batch_first=True,
            norm_first=True,
        )
        self.bev_encoder = nn.TransformerEncoder(enc_layer, num_layers=cfg.scene_encoder_layers)
        self.bev_norm = nn.LayerNorm(cfg.d_model)

        self.past_input_proj = nn.Linear(2, cfg.d_model)
        self.past_pos_emb = nn.Parameter(torch.zeros(1, cfg.past_len_max, cfg.d_model))
        past_layer = nn.TransformerEncoderLayer(
            d_model=cfg.d_model,
            nhead=cfg.n_heads,
            dim_feedforward=cfg.mlp_dim,
            dropout=cfg.scene_dropout,
            batch_first=True,
            norm_first=True,
        )
        self.past_encoder = nn.TransformerEncoder(past_layer, num_layers=cfg.past_num_layers)
        self.past_norm = nn.LayerNorm(cfg.d_model)
        self.past_token_mlp = nn.Sequential(
            nn.Linear(cfg.d_model * 2, cfg.past_encoder_dim),
            nn.ReLU(inplace=True),
            nn.Linear(cfg.past_encoder_dim, cfg.d_model),
        )

        if cfg.use_intent:
            self.intent_emb = nn.Embedding(cfg.num_intents, cfg.intent_emb_dim)
            self.intent_proj = nn.Sequential(
                nn.Linear(cfg.intent_emb_dim, cfg.d_model),
                nn.ReLU(inplace=True),
                nn.Linear(cfg.d_model, cfg.d_model),
            )
        else:
            self.intent_emb = None
            self.intent_proj = None

        self.scene_fuse = nn.Sequential(
            nn.Linear(cfg.d_model * 3, cfg.mlp_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(cfg.scene_dropout),
            nn.Linear(cfg.mlp_dim, cfg.d_model),
        )
        self.scene_gate = nn.Sequential(
            nn.Linear(cfg.d_model * 3, cfg.d_model),
            nn.Sigmoid(),
        )

    def forward(self, x, past_xy, intent):
        feat = self.stem(x)
        feat = self.layer1(feat)
        feat = self.layer2(feat)
        feat = self.layer3(feat)
        feat = self.layer4(feat)

        bev = self.proj(self.pool(feat))                 # [B, d_model, 8, 8]
        bev_tokens = bev.flatten(2).transpose(1, 2)     # [B, 64, d_model]
        bev_tokens = self.bev_encoder(bev_tokens + self.bev_pos_emb)
        bev_tokens = self.bev_norm(bev_tokens)

        past_tokens = self.past_input_proj(past_xy) + self.past_pos_emb
        past_tokens = self.past_encoder(past_tokens)
        past_tokens = self.past_norm(past_tokens)
        past_last = past_tokens[:, -1]
        past_mean = past_tokens.mean(dim=1)
        past_token = self.past_token_mlp(torch.cat([past_last, past_mean], dim=-1))

        if self.cfg.use_intent:
            intent_token = self.intent_proj(self.intent_emb(intent))
        else:
            intent_token = torch.zeros_like(past_token)

        global_bev = bev_tokens.mean(dim=1)
        fused_input = torch.cat([global_bev, past_token, intent_token], dim=-1)
        base_scene = self.scene_fuse(fused_input)
        gate = self.scene_gate(fused_input)
        scene_token = base_scene * gate + global_bev * (1.0 - gate)

        return {
            "bev_tokens": bev_tokens,          # [B, 64, d_model]
            "scene_token": scene_token,        # [B, d_model]
            "past_token": past_token,          # [B, d_model]
            "intent_token": intent_token,      # [B, d_model]
        }


In [166]:
class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, x):
        device = x.device
        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = x[:, None] * emb[None, :]
        emb = torch.cat([emb.sin(), emb.cos()], dim=-1)
        return emb


class TrajEncoder(nn.Module):
    def __init__(self, future_len, d_model):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(future_len * 2, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, d_model),
            nn.LayerNorm(d_model),
        )

    def forward(self, traj_xy):
        # traj_xy: [B, K, T, 2]
        b, k, t, c = traj_xy.shape
        x = traj_xy.reshape(b, k, t * c)
        return self.mlp(x)


class DiffusionPlannerHead(nn.Module):
    def __init__(self, anchors_np, cfg: CFG):
        super().__init__()
        self.cfg = cfg
        self.K = anchors_np.shape[0]
        self.T = anchors_np.shape[1]

        self.register_buffer("anchors", torch.tensor(anchors_np, dtype=torch.float32), persistent=True)

        self.scheduler = DDIMScheduler(
            num_train_timesteps=cfg.num_train_timesteps,
            beta_schedule="scaled_linear",
            prediction_type="sample",
        )

        self.traj_encoder = TrajEncoder(cfg.future_len, cfg.d_model)
        self.time_mlp = nn.Sequential(
            SinusoidalPosEmb(cfg.d_model),
            nn.Linear(cfg.d_model, cfg.d_model * 4),
            nn.Mish(),
            nn.Linear(cfg.d_model * 4, cfg.d_model),
        )

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=cfg.d_model,
            nhead=cfg.n_heads,
            dim_feedforward=cfg.mlp_dim,
            dropout=cfg.scene_dropout,
            batch_first=True,
            norm_first=True,
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=cfg.n_decoder_layers)

        self.reg_head = nn.Sequential(
            nn.Linear(cfg.d_model, cfg.mlp_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(cfg.scene_dropout),
            nn.Linear(cfg.mlp_dim, cfg.future_len * 2),
        )

        self.cls_head = nn.Sequential(
            nn.Linear(cfg.d_model, cfg.mlp_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(cfg.scene_dropout),
            nn.Linear(cfg.mlp_dim, 1),
        )

    def _add_noise_to_anchors_train(self, batch_size, device):
        anchors = self.anchors[None].repeat(batch_size, 1, 1, 1)  # [B,K,T,2]
        timesteps = torch.randint(
            0, self.cfg.train_max_timestep, (batch_size,), device=device
        )
        noise = torch.randn_like(anchors)
        noisy = self.scheduler.add_noise(anchors, noise, timesteps)
        return anchors, noisy, timesteps

    def _scene_memory(self, bev_tokens, scene_token):
        # bev_tokens: [B,64,d], scene_token: [B,d]
        return torch.cat([bev_tokens, scene_token[:, None, :]], dim=1)

    def _compute_soft_anchor_targets(self, gt):
        # gt: [B, 1, T, 2]
        dists = torch.norm(self.anchors[None] - gt, dim=-1).mean(dim=-1)  # [B, K]
        soft_targets = F.softmax(-dists / self.cfg.soft_target_temperature, dim=1)
        return soft_targets

    def forward_train(self, bev_tokens, scene_token, targets):
        device = bev_tokens.device
        bs = bev_tokens.shape[0]

        anchors, noisy, timesteps = self._add_noise_to_anchors_train(bs, device)
        traj_tokens = self.traj_encoder(noisy)                    # [B,K,d]
        time_emb = self.time_mlp(timesteps)[:, None, :]          # [B,1,d]
        query = traj_tokens + scene_token[:, None, :] + time_emb

        memory = self._scene_memory(bev_tokens, scene_token)
        dec = self.decoder(query, memory)                        # [B,K,d]

        delta = self.reg_head(dec).reshape(bs, self.K, self.T, 2)
        pred = noisy + delta
        scores = self.cls_head(dec).squeeze(-1)                  # [B,K]

        gt = targets["future_xy"][:, None, :, :]                 # [B,1,T,2]
        pos_id = targets["anchor_id"]                            # [B]

        soft_targets = self._compute_soft_anchor_targets(gt)
        log_probs = F.log_softmax(scores, dim=1)
        cls_loss = -(soft_targets * log_probs).sum(dim=1).mean()

        pos_pred = pred[torch.arange(bs, device=device), pos_id] # [B,T,2]
        reg_main = F.smooth_l1_loss(
            pos_pred,
            targets["future_xy"],
            beta=self.cfg.smooth_l1_beta,
        )
        reg_end = F.smooth_l1_loss(
            pos_pred[:, -1],
            targets["future_xy"][:, -1],
            beta=self.cfg.smooth_l1_beta,
        )
        reg_loss = reg_main + self.cfg.endpoint_loss_weight * reg_end

        total_loss = self.cfg.cls_loss_weight * cls_loss + self.cfg.reg_loss_weight * reg_loss

        best_id = scores.argmax(dim=1)
        best_pred = pred[torch.arange(bs, device=device), best_id]

        return {
            "trajectory": best_pred,
            "all_traj": pred,
            "scores": scores,
            "loss": total_loss,
            "loss_cls": cls_loss.detach(),
            "loss_reg": reg_loss.detach(),
            "pos_pred": pos_pred.detach(),
        }

    @torch.no_grad()
    def forward_test(self, bev_tokens, scene_token):
        device = bev_tokens.device
        bs = bev_tokens.shape[0]

        self.scheduler.set_timesteps(self.cfg.num_train_timesteps, device=device)
        anchors = self.anchors[None].repeat(bs, 1, 1, 1)

        sample = anchors.clone()
        noise = torch.randn_like(sample)
        trunc_ts = torch.full((bs,), self.cfg.infer_start_timestep, dtype=torch.long, device=device)
        sample = self.scheduler.add_noise(sample, noise, trunc_ts)

        step_ratio = 20 / self.cfg.infer_steps
        roll_timesteps = (np.arange(0, self.cfg.infer_steps) * step_ratio).round()[::-1].copy().astype(np.int64)
        roll_timesteps = torch.from_numpy(roll_timesteps).to(device)

        memory = self._scene_memory(bev_tokens, scene_token)

        scores = None
        pred = None
        for k in roll_timesteps:
            traj_tokens = self.traj_encoder(sample)
            tt = torch.tensor([int(k.item())], device=device).expand(bs)
            time_emb = self.time_mlp(tt)[:, None, :]
            query = traj_tokens + scene_token[:, None, :] + time_emb

            dec = self.decoder(query, memory)
            delta = self.reg_head(dec).reshape(bs, self.K, self.T, 2)
            pred = sample + delta
            scores = self.cls_head(dec).squeeze(-1)

            model_output = pred
            sample = self.scheduler.step(
                model_output=model_output,
                timestep=k,
                sample=sample
            ).prev_sample

        best_id = scores.argmax(dim=1)
        best_pred = pred[torch.arange(bs, device=device), best_id]

        return {
            "trajectory": best_pred,
            "all_traj": pred,
            "scores": scores,
        }


In [167]:
class WaymoDiffusionPlanner(nn.Module):
    def __init__(self, in_ch, anchors_np, cfg: CFG):
        super().__init__()
        self.encoder = SimpleSceneEncoder(in_ch, cfg)
        self.head = DiffusionPlannerHead(anchors_np, cfg)
        self.cfg = cfg

    def forward(self, batch):
        enc = self.encoder(batch["x"], batch["past_xy"], batch["intent"])

        if self.training:
            out = self.head.forward_train(
                bev_tokens=enc["bev_tokens"],
                scene_token=enc["scene_token"],
                targets=batch,
            )
        else:
            out = self.head.forward_test(
                bev_tokens=enc["bev_tokens"],
                scene_token=enc["scene_token"],
            )

        return out

In [168]:
in_ch = train_ds[0]["x"].shape[0]
model = WaymoDiffusionPlanner(in_ch=in_ch, anchors_np=anchors, cfg=cfg).to(device)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("in_ch:", in_ch)
print("params:", n_params)
print("trainable:", n_trainable)

in_ch: 8
params: 3821081
trainable: 3821081


In [169]:
def move_batch_to_device(batch, device):
    out = {}
    for k, v in batch.items():
        if torch.is_tensor(v):
            out[k] = v.to(device, non_blocking=True)
        else:
            out[k] = v
    return out

batch = next(iter(train_loader))
batch = move_batch_to_device(batch, device)

model.train()
out = model(batch)

for k, v in out.items():
    if torch.is_tensor(v):
        print(k, tuple(v.shape), v.dtype)
    else:
        print(k, type(v))

trajectory (32, 20, 2) torch.float32
all_traj (32, 20, 20, 2) torch.float32
scores (32, 20) torch.float32
loss () torch.float32
loss_cls () torch.float32
loss_reg () torch.float32
pos_pred (32, 20, 2) torch.float32


In [170]:
@torch.no_grad()
def compute_ade_fde(pred_xy, gt_xy):
    # pred_xy, gt_xy: [B,T,2]
    l2 = torch.norm(pred_xy - gt_xy, dim=-1)     # [B,T]
    ade = l2.mean().item()
    fde = l2[:, -1].mean().item()
    return ade, fde

@torch.no_grad()
def compute_anchor_cls_acc(scores, anchor_id):
    pred = scores.argmax(dim=1)
    acc = (pred == anchor_id).float().mean().item()
    return acc

In [171]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.lr,
    weight_decay=cfg.weight_decay,
)

scaler = torch.cuda.amp.GradScaler(enabled=(cfg.mixed_precision and device.type == "cuda"))

def save_checkpoint(path, model, optimizer, epoch, best_val_fde, cfg):
    torch.save({
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "epoch": epoch,
        "best_val_fde": best_val_fde,
        "cfg": asdict(cfg),
    }, path)

def append_jsonl(path, row):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

train_log_path = Path(cfg.work_dir) / "train_log.jsonl"
val_log_path = Path(cfg.work_dir) / "val_log.jsonl"
best_ckpt_path = Path(cfg.work_dir) / "best_model.pt"
last_ckpt_path = Path(cfg.work_dir) / "last_model.pt"

In [172]:
def train_one_epoch(
    model,
    loader,
    optimizer,
    scaler,
    device,
    epoch,
    cfg,
    train_log_path=None,
    midepoch_ckpt_path=None,
    best_val_fde=None,
):
    model.train()

    running_loss = 0.0
    running_cls = 0.0
    running_reg = 0.0
    running_ade = 0.0
    running_fde = 0.0
    running_acc = 0.0
    n_batches = 0

    total_steps = len(loader)
    mid_step = max(1, total_steps // 2)

    pbar = tqdm(loader, desc=f"train epoch {epoch}", leave=False)

    for step, batch in enumerate(pbar, start=1):
        batch = move_batch_to_device(batch, device)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(cfg.mixed_precision and device.type == "cuda")):
            out = model(batch)
            loss = out["loss"]

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip_norm)
        scaler.step(optimizer)
        scaler.update()

        ade, fde = compute_ade_fde(out["trajectory"], batch["future_xy"])
        acc = compute_anchor_cls_acc(out["scores"], batch["anchor_id"])

        running_loss += loss.item()
        running_cls += out["loss_cls"].item()
        running_reg += out["loss_reg"].item()
        running_ade += ade
        running_fde += fde
        running_acc += acc
        n_batches += 1

        avg_metrics = {
            "epoch": epoch,
            "step": step,
            "steps_total": total_steps,
            "progress": step / total_steps,
            "loss": running_loss / n_batches,
            "loss_cls": running_cls / n_batches,
            "loss_reg": running_reg / n_batches,
            "ade": running_ade / n_batches,
            "fde": running_fde / n_batches,
            "anchor_acc": running_acc / n_batches,
        }

        if step % cfg.log_every == 0 or step == mid_step or step == total_steps:
            pbar.set_postfix({
                "loss": f"{avg_metrics['loss']:.4f}",
                "cls": f"{avg_metrics['loss_cls']:.4f}",
                "reg": f"{avg_metrics['loss_reg']:.4f}",
                "ade": f"{avg_metrics['ade']:.4f}",
                "fde": f"{avg_metrics['fde']:.4f}",
                "acc": f"{avg_metrics['anchor_acc']:.4f}",
            })

            if train_log_path is not None:
                log_row = {"type": "train_step", **avg_metrics}
                append_jsonl(train_log_path, log_row)

        if step == mid_step:
            print(f"\n[epoch {epoch} | mid-epoch] train:",
                  {k: round(v, 4) if isinstance(v, float) else v for k, v in avg_metrics.items()})

            if midepoch_ckpt_path is not None:
                save_checkpoint(
                    midepoch_ckpt_path,
                    model,
                    optimizer,
                    epoch,
                    best_val_fde if best_val_fde is not None else float("inf"),
                    cfg,
                )
                print(f"  -> saved MID-EPOCH checkpoint to {midepoch_ckpt_path}")

    metrics = {
        "epoch": epoch,
        "loss": running_loss / n_batches,
        "loss_cls": running_cls / n_batches,
        "loss_reg": running_reg / n_batches,
        "ade": running_ade / n_batches,
        "fde": running_fde / n_batches,
        "anchor_acc": running_acc / n_batches,
    }
    return metrics

def validate(model, loader, device, epoch):
    model.eval()

    running_ade = 0.0
    running_fde = 0.0
    n_batches = 0

    pbar = tqdm(loader, desc=f"val epoch {epoch}", leave=False)

    for batch in pbar:
        batch = move_batch_to_device(batch, device)
        out = model(batch)

        ade, fde = compute_ade_fde(out["trajectory"], batch["future_xy"])

        running_ade += ade
        running_fde += fde
        n_batches += 1

    metrics = {
        "epoch": epoch,
        "ade": running_ade / n_batches,
        "fde": running_fde / n_batches,
    }
    return metrics

In [173]:
@torch.no_grad()
def validate(model, loader, device, epoch):
    model.eval()

    running_ade = 0.0
    running_fde = 0.0
    n_batches = 0

    pbar = tqdm(loader, desc=f"val epoch {epoch}", leave=False)

    for batch in pbar:
        batch = move_batch_to_device(batch, device)
        out = model(batch)

        ade, fde = compute_ade_fde(out["trajectory"], batch["future_xy"])

        running_ade += ade
        running_fde += fde
        n_batches += 1

    metrics = {
        "epoch": epoch,
        "ade": running_ade / n_batches,
        "fde": running_fde / n_batches,
    }
    return metrics

In [174]:
train_log_path = Path(cfg.work_dir) / "train_log.jsonl"
val_log_path = Path(cfg.work_dir) / "val_log.jsonl"
best_ckpt_path = Path(cfg.work_dir) / "best_model.pt"
last_ckpt_path = Path(cfg.work_dir) / "last_model.pt"
last_midepoch_ckpt_path = Path(cfg.work_dir) / "last_midepoch.pt"
error_log_path = Path(cfg.work_dir) / "error_log.txt"
warning_log_path = Path(cfg.work_dir) / "warning_log.txt"

print(train_log_path)
print(val_log_path)
print(best_ckpt_path)
print(last_ckpt_path)
print(last_midepoch_ckpt_path)
print(error_log_path)
print(warning_log_path)

precomputed_waymo_e2e/training_runs_20/diffusion_planner_v1/train_log.jsonl
precomputed_waymo_e2e/training_runs_20/diffusion_planner_v1/val_log.jsonl
precomputed_waymo_e2e/training_runs_20/diffusion_planner_v1/best_model.pt
precomputed_waymo_e2e/training_runs_20/diffusion_planner_v1/last_model.pt
precomputed_waymo_e2e/training_runs_20/diffusion_planner_v1/last_midepoch.pt
precomputed_waymo_e2e/training_runs_20/diffusion_planner_v1/error_log.txt
precomputed_waymo_e2e/training_runs_20/diffusion_planner_v1/warning_log.txt


In [175]:
import warnings
import traceback
from datetime import datetime

def append_text(path, text):
    with open(path, "a", encoding="utf-8") as f:
        f.write(text + "\n")

def log_warning_to_file(message, category, filename, lineno, file=None, line=None):
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    text = f"[{ts}] {category.__name__}: {message} | {filename}:{lineno}"
    append_text(warning_log_path, text)

warnings.showwarning = log_warning_to_file

In [ ]:
import gc

best_val_fde = float("inf")
history = {
    "train": [],
    "val": [],
}

try:
    for epoch in range(1, cfg.epochs + 1):
        t0 = time.time()

        train_metrics = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            scaler=scaler,
            device=device,
            epoch=epoch,
            cfg=cfg,
            train_log_path=train_log_path,
            midepoch_ckpt_path=last_midepoch_ckpt_path,
            best_val_fde=best_val_fde,
        )
        history["train"].append(train_metrics)

        append_jsonl(train_log_path, {"type": "train_epoch", **train_metrics})

        print(f"\n[epoch {epoch}] train:",
              {k: round(v, 4) if isinstance(v, float) else v for k, v in train_metrics.items()})

        if epoch % cfg.val_every == 0:
            val_metrics = validate(model, val_loader, device, epoch)
            history["val"].append(val_metrics)
            append_jsonl(val_log_path, {"type": "val_epoch", **val_metrics})

            print(f"[epoch {epoch}] val:",
                  {k: round(v, 4) if isinstance(v, float) else v for k, v in val_metrics.items()})

            if val_metrics["fde"] < best_val_fde:
                best_val_fde = val_metrics["fde"]
                save_checkpoint(best_ckpt_path, model, optimizer, epoch, best_val_fde, cfg)
                print(f"  -> saved BEST checkpoint to {best_ckpt_path}")

        save_checkpoint(last_ckpt_path, model, optimizer, epoch, best_val_fde, cfg)

        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()

        dt = time.time() - t0
        print(f"[epoch {epoch}] time: {dt/60:.2f} min")

except Exception:
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    append_text(error_log_path, "\n" + "=" * 100)
    append_text(error_log_path, f"[{ts}] TRAINING CRASH")
    append_text(error_log_path, traceback.format_exc())

    # пробуем сохранить аварийный чекпоинт
    try:
        crash_ckpt_path = Path(cfg.work_dir) / "crash_model.pt"
        save_checkpoint(crash_ckpt_path, model, optimizer, epoch, best_val_fde, cfg)
        append_text(error_log_path, f"Crash checkpoint saved to: {crash_ckpt_path}")
    except Exception:
        append_text(error_log_path, "Failed to save crash checkpoint.")
        append_text(error_log_path, traceback.format_exc())

    raise

train epoch 1:  50%|████▉     | 6494/12989 [45:44<49:27,  2.19it/s, loss=1.5089, cls=0.1031, reg=0.7029, ade=2.7284, fde=6.7089, acc=0.5215]   


[epoch 1 | mid-epoch] train: {'epoch': 1, 'step': 6494, 'steps_total': 12989, 'progress': 0.5, 'loss': 1.5089, 'loss_cls': 0.1031, 'loss_reg': 0.7029, 'ade': 2.7284, 'fde': 6.7089, 'anchor_acc': 0.5215}
  -> saved MID-EPOCH checkpoint to precomputed_waymo_e2e/training_runs_20/diffusion_planner_v1/last_midepoch.pt



[epoch 1] train: {'epoch': 1, 'loss': 1.4367, 'loss_cls': 0.0961, 'loss_reg': 0.6703, 'ade': 2.423, 'fde': 6.0714, 'anchor_acc': 0.5527}


[epoch 1] val: {'epoch': 1, 'ade': 9.7655, 'fde': 24.68}
  -> saved BEST checkpoint to precomputed_waymo_e2e/training_runs_20/diffusion_planner_v1/best_model.pt
[epoch 1] time: 114.12 min


train epoch 2:  50%|████▉     | 6494/12989 [44:55<46:24,  2.33it/s, loss=1.3211, cls=0.0858, reg=0.6177, ade=2.0078, fde=5.1931, acc=0.5991]  

...


[epoch 4] train: {'epoch': 4, 'loss': 1.2171, 'loss_cls': 0.0792, 'loss_reg': 0.5689, 'ade': 1.7719, 'fde': 4.6406, 'anchor_acc': 0.6329}


[epoch 4] val: {'epoch': 4, 'ade': 9.3094, 'fde': 22.6864}
[epoch 4] time: 111.98 min


train epoch 5:  50%|████▉     | 6494/12989 [44:45<43:58,  2.46it/s, loss=1.1827, cls=0.0777, reg=0.5525, ade=1.7146, fde=4.4927, acc=0.6406]  


[epoch 5 | mid-epoch] train: {'epoch': 5, 'step': 6494, 'steps_total': 12989, 'progress': 0.5, 'loss': 1.1827, 'loss_cls': 0.0777, 'loss_reg': 0.5525, 'ade': 1.7146, 'fde': 4.4927, 'anchor_acc': 0.6406}
  -> saved MID-EPOCH checkpoint to precomputed_waymo_e2e/training_runs_20/diffusion_planner_v1/last_midepoch.pt


train epoch 5:  52%|█████▏    | 6792/12989 [46:45<32:46,  3.15it/s, loss=1.1820, cls=0.0777, reg=0.5521, ade=1.7143, fde=4.4920, acc=0.6407]

In [ ]:
ckpt = torch.load(best_ckpt_path, map_location=device)
model.load_state_dict(ckpt["model_state"])
print("loaded best checkpoint from epoch:", ckpt["epoch"])
print("best val fde:", ckpt["best_val_fde"])

In [ ]:
@torch.no_grad()
def show_val_predictions(model, ds, n=5):
    model.eval()

    idxs = np.random.choice(len(ds), size=n, replace=False)

    for idx in idxs:
        sample = ds[idx]
        batch = {
            "x": sample["x"][None].to(device),
            "past_xy": sample["past_xy"][None].to(device),
            "future_xy": sample["future_xy"][None].to(device),
            "intent": sample["intent"][None].to(device),
            "sample_id": sample["sample_id"][None].to(device),
        }

        out = model(batch)
        pred = out["trajectory"][0].cpu().numpy()
        gt = sample["future_xy"].numpy()
        past = sample["past_xy"].numpy()
        scores = out["scores"][0].cpu().numpy()
        best_mode = int(scores.argmax())
        anchor = anchors[best_mode]

        plt.figure(figsize=(6,6))
        plt.plot(past[:,0], past[:,1], marker="o", label="past")
        plt.plot(gt[:,0], gt[:,1], marker="o", label="gt_future")
        plt.plot(anchor[:,0], anchor[:,1], marker="o", label=f"anchor_{best_mode}")
        plt.plot(pred[:,0], pred[:,1], marker="o", label="pred")
        plt.axhline(0, color="gray", linewidth=1)
        plt.axvline(0, color="gray", linewidth=1)
        plt.grid(True, alpha=0.3)
        plt.axis("equal")
        plt.legend()
        plt.title(f"sample_id={sample['sample_id'].item()} intent={sample['intent'].item()}")
        plt.show()

show_val_predictions(model, val_ds, n=5)


In [ ]:
ckpt_last = torch.load(last_ckpt_path, map_location="cpu")
print("last checkpoint epoch:", ckpt_last["epoch"])

if best_ckpt_path.exists():
    ckpt_best = torch.load(best_ckpt_path, map_location="cpu")
    print("best checkpoint epoch:", ckpt_best["epoch"])
    print("best val fde:", ckpt_best["best_val_fde"])